# Mushroom — vergelijking en kritische conclusie

**Bijdrage en AI:** Codex maakte dit concept op verzoek van de gebruiker.
De mapnaam is geen bevestiging van Milans persoonlijke auteurschap of review.
De oorspronkelijke teamoplossing blijft behouden. De kritiek en lesverwijzingen
staan in [REVIEW_FINAL.md](REVIEW_FINAL.md).

Dit is ontwikkelonderzoek. De 900 al bekende ontwikkeltestrecords worden voor
dit experiment niet gescoord. De overblijvende projectdata zijn ook eerder
gebruikt; er ontstaat dus geen nieuwe onafhankelijke eindtest.

## Buitenfoldmetrics, geen testselectie
Elke variant krijgt precies één voorspelling voor ieder van de 4.100 rijen wanneer die rij buiten de training ligt. Bij default gebruiken we de echte sklearn-labels; bij de recallpolicy geldt score >= de binnen training gekozen drempel.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'MushroomFinal/mushroom_project_dataset.csv').exists())
HERE = ROOT/'solutions/milan'
sys.path.insert(0, str(HERE))
from mushroom_alternative import development_data, pipeline, GRID, scores
X, y, ids, folds = development_data()
summary = pd.read_csv(HERE/'results/summary.csv')
fold_metrics = pd.read_csv(HERE/'results/outer_fold_metrics.csv')
pred = pd.read_csv(HERE/'results/outer_oof_predictions.csv', float_precision='round_trip')
assert len(pred) == 4*4100
for name, group in pred.groupby('model'):
    assert group.source_row.is_unique and set(group.source_row) == set(ids)
    for fold, part in group.groupby('fold'):
        calculated = scores(part.true_class, part.probability_p, prediction=part.prediction)
        stored = fold_metrics[(fold_metrics.model == name) & (fold_metrics.fold == fold)].iloc[0]
        for metric, value in calculated.items():
            assert abs(value - stored[metric]) < 1e-12, (name, fold, metric)
print(summary[['model','macro_f1_mean','macro_f1_sd','recall_p_mean','precision_p_mean','specificity_e_mean','fn_total','fp_total']].round(4).to_string(index=False))


                        model  macro_f1_mean  macro_f1_sd  recall_p_mean  precision_p_mean  specificity_e_mean  fn_total  fp_total
      rf300_reference_default         0.7619       0.0215         0.6044            0.7862              0.8995       614       256
rf300_reference_recall_target         0.5652       0.0160         0.9066            0.4661              0.3670       145      1613
             rf_tuned_default         0.7629       0.0269         0.6179            0.7743              0.8901       593       280
       rf_tuned_recall_target         0.5808       0.0091         0.9033            0.4748              0.3913       150      1551


## De afweging zichtbaar
De grafiek gebruikt dezelfde buitenfolds. Meer recall kan samengaan met meer valse alarmen en lagere macro F1. SD over folds is geen betrouwbaarheidsinterval. De 90%-binnenregel garandeert geen 90% buitenrecall.

In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
labels = ['RF300\nstandaard','RF300\nrecallregel','RF getuned\nstandaard','RF getuned\nrecallregel']
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')
pos = np.arange(len(summary))
axes[0].bar(pos-.18, summary.macro_f1_mean, width=.36, label='Macro F1', color='#24688e')
axes[0].bar(pos+.18, summary.recall_p_mean, width=.36, label='Recall giftig', color='#d47b20')
axes[0].set(xticks=pos, xticklabels=labels, ylim=(0,1.05), title='Gemiddelde over vijf buitenfolds', ylabel='Score')
axes[0].legend()
axes[1].bar(pos-.18, summary.fn_total, width=.36, label='Giftig gemist (FN)', color='#b74740')
axes[1].bar(pos+.18, summary.fp_total, width=.36, label='Vals alarm (FP)', color='#8d9198')
axes[1].set(xticks=pos, xticklabels=labels, title='Fouten op dezelfde 4.100 ontwikkelrijen', ylabel='Aantal')
axes[1].legend()
fig.savefig(HERE/'results/vergelijking.png', dpi=150)
plt.close(fig)
print('Grafiek opgeslagen: results/vergelijking.png')


Grafiek opgeslagen: results/vergelijking.png


![Buitenfoldvergelijking](results/vergelijking.png)

## Betekenis van de uitkomst
Lees [RESULTATEN.md](RESULTATEN.md) voor het gemeten oordeel. Een kleine afwijking op deze reeds gebruikte ontwikkelingsdata bewijst geen algemene superioriteit. De sklearn 1.6.1-referentie en kandidaat zijn wel eerlijk binnen één omgeving vergeleken. Vergelijk onze getallen daarom niet alsof het een nieuwe score op de 900 ontwikkeltestrecords is. Het experiment maakt geen website-artifact en verandert geen hosting.

In [3]:
print('Onveranderde bronhash:', json.loads((HERE/'results/protocol.json').read_text())['source_sha256_lf'])
print('Nieuwe testrecords gescoord:', json.loads((HERE/'results/run.json').read_text())['test_rows_scored'])
print('Nieuwe deployment gemaakt:', json.loads((HERE/'results/run.json').read_text())['new_deployment_artifact_created'])


Onveranderde bronhash: 1f1a25f2f330458ed95ce9f6ffe3241582312a42cb21797a2b7bd5e5ad281114
Nieuwe testrecords gescoord: 0
Nieuwe deployment gemaakt: False
